# Voyage Analytics — Data Ingestion

**Notebook:** `02_Data_Ingestion` · **Phase:** Week 1–2 (Data Management)
**Depends on:** `src/data_ingestion`, `src/utils` · **Produces:** `data/interim/`, `data/processed/`

---

This notebook is a thin **walkthrough of the production ingestion module** in `src/`. All logic lives
in the package so the same code runs from the notebook, a script (`python -m src.data_ingestion`),
an Airflow task, or the test suite — the notebook only *drives and inspects* it.

> **Performance note.** The heavy single pass over the 271k-row raw data is done by the `src` pipeline.
> This notebook reads the compact **processed CSVs** it produces, so it stays light on memory
> (open and read freely; no full-dataset re-processing here).

> **Format note.** Interim/processed tables are written as **CSV** (human-readable, Excel-friendly).
> Because CSV stores everything as text, always load them through the typed readers
> `load_interim()` / `load_processed()`, which restore dates and categoricals from `CONFIG`.

### Pipeline
```
data/raw/*.csv
   │  load_raw()            read the three CSVs  (already gated by src.validation)
   ├─ clean_*()   ─────────▶ data/interim/    *.csv  (parsed, typed, leg-tagged, feature-enriched)
   └─ build_*()   ─────────▶ data/processed/  *.csv  (trips · users_features · flight_price_model)
```


## 0. Setup — make `src` importable

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Add repo root to path so `import src...` works from notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.utils import CONFIG, PATHS
print("project root:", ROOT)
print("raw       :", PATHS.raw)
print("interim   :", PATHS.interim)
print("processed :", PATHS.processed)

project root: F:\Final Project Labmentix\voyage-analytics
raw       : F:\Final Project Labmentix\voyage-analytics\data\raw
interim   : F:\Final Project Labmentix\voyage-analytics\data\interim
processed : F:\Final Project Labmentix\voyage-analytics\data\processed


## 1. Load raw

Ingestion assumes its input has already been gated by the **validation stage**
(`src/validation`, notebook `05_Data_Validation`), so this module only transforms.

In [2]:
from src.data_ingestion import load_raw

raw = load_raw()
for name, df in raw.items():
    print(f"{name:<8} {df.shape}")

17:25:25 | INFO    | ingest | loaded flights  (271888, 10) from flights.csv


17:25:26 | INFO    | ingest | loaded hotels   (40552, 8) from hotels.csv


17:25:26 | INFO    | ingest | loaded users    (1340, 5) from users.csv


flights  (271888, 10)
hotels   (40552, 8)
users    (1340, 5)


## 2. Cleaning → `data/interim`

The `clean_*` functions parse dates, tag outbound/return legs, add calendar + route + unit-price
features, normalise gender, and downcast to memory-friendly dtypes. Preview the results (no files
written in this cell — persistence happens in the full-pipeline call below).

In [3]:
from src.data_ingestion import clean_flights, clean_hotels, clean_users

flights_c = clean_flights(raw["flights"])
hotels_c  = clean_hotels(raw["hotels"])
users_c   = clean_users(raw["users"])

print("flights_c:", flights_c.shape, "->", list(flights_c.columns))
display(flights_c.head(3))
print("memory (MB):", round(flights_c.memory_usage(deep=True).sum()/1e6, 1),
      "(raw was ~92 MB — categoricals shrink it)")

flights_c: (271888, 16) -> ['travelCode', 'userCode', 'from', 'to', 'flightType', 'price', 'time', 'distance', 'agency', 'date', 'leg', 'year', 'month', 'day_of_week', 'route', 'price_per_km']


,travelCode,userCode,from,to,flightType,price,time,distance,agency,date,leg,year,month,day_of_week,route,price_per_km
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,2019-09-26,outbound,2019,9,Thursday,Recife (PE) -> Florianopolis (SC),2.1202
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,2019-09-30,return,2019,9,Monday,Florianopolis (SC) -> Recife (PE),1.9102
2,1,0,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,2019-10-03,outbound,2019,10,Thursday,Brasilia (DF) -> Florianopolis (SC),2.3331


memory (MB): 18.0 (raw was ~92 MB — categoricals shrink it)


In [4]:
print("users_c — gender normalised + gender_known flag:")
display(users_c.head(3))
print("gender_known share: {:.1%}".format(users_c['gender_known'].mean()))

users_c — gender normalised + gender_known flag:


,code,company,name,gender,age,gender_known
0,0,4You,Roy Braun,male,21,True
1,1,4You,Joseph Holsten,male,37,True
2,2,4You,Wilma Mcinnis,female,48,True


gender_known share: 67.2%


## 3. Feature tables → `data/processed`

Three modelling-ready tables:
- **`trips`** — one row per round trip (legs collapsed) with hotel attach + spend.
- **`users_features`** — user-level RFM + behaviour joined to demographics (home city, recency, spend).
- **`flight_price_model`** — the regressor's feature table (`time` dropped: r=0.99999 with `distance`).

In [5]:
from src.data_ingestion import build_trips, build_user_features, build_flight_price_table
from src.data_ingestion import _attach_hotels

trips = _attach_hotels(build_trips(flights_c), hotels_c)
udf   = build_user_features(users_c, trips, flights_c)
price = build_flight_price_table(flights_c)

print("trips:", trips.shape); display(trips.head(3))

trips: (135944, 15)


,travelCode,user,origin,dest,flightType,agency,depart,return_date,distance,flight_cost,year,trip_nights,hotel_cost,has_hotel,trip_spend
0,0,0,Recife (PE),Florianopolis (SC),firstClass,FlyingDrops,2019-09-26,2019-09-30,676.53,2726.67,2019,4,1252.08,True,3978.75
1,1,0,Brasilia (DF),Florianopolis (SC),firstClass,CloudFy,2019-10-03,2019-10-04,637.56,2614.88,2019,1,0.00,False,2614.88
2,2,0,Aracaju (SE),Salvador (BH),firstClass,CloudFy,2019-10-10,2019-10-12,830.86,3215.97,2019,2,526.82,True,3742.79


In [6]:
print("users_features:", udf.shape)
display(udf[["code","company","gender","age","home_city","n_trips",
             "total_spend","recency_days","hotel_attach_rate","is_active"]].head(3))
print("\nflight_price_model:", price.shape, "->", list(price.columns))
display(price.head(3))

users_features: (1340, 19)


,code,company,gender,age,home_city,n_trips,total_spend,recency_days,hotel_attach_rate,is_active
0,0,4You,male,21,Brasilia (DF),89.0,192068.69,781.0,0.3034,True
1,1,4You,male,37,Aracaju (SE),6.0,10213.50,1362.0,0.3333,True
2,2,4You,female,48,Aracaju (SE),131.0,257701.64,487.0,0.2748,True



flight_price_model: (271888, 10) -> ['from', 'to', 'route', 'flightType', 'agency', 'distance', 'year', 'month', 'day_of_week', 'price']


,from,to,route,flightType,agency,distance,year,month,day_of_week,price
0,Recife (PE),Florianopolis (SC),Recife (PE) -> Florianopolis (SC),firstClass,FlyingDrops,676.53,2019,9,Thursday,1434.38
1,Florianopolis (SC),Recife (PE),Florianopolis (SC) -> Recife (PE),firstClass,FlyingDrops,676.53,2019,9,Monday,1292.29
2,Brasilia (DF),Florianopolis (SC),Brasilia (DF) -> Florianopolis (SC),firstClass,CloudFy,637.56,2019,10,Thursday,1487.52


## 4. Run the full pipeline (writes all files)

One call runs clean → feature-build → persist. It writes **CSV** to
`data/interim/` and `data/processed/` and returns the frames and the written paths.
This is the same function `python -m src.data_ingestion` executes.

In [7]:
from src.data_ingestion import run_pipeline

result = run_pipeline(parquet=False)   # CSV outputs
print("\nfiles written:")
for p in result["written"]:
    print("  ", p.relative_to(PATHS.root))

17:25:27 | INFO    | ingest | === Voyage Analytics ingestion started ===


17:25:27 | INFO    | ingest | loaded flights  (271888, 10) from flights.csv


17:25:27 | INFO    | ingest | loaded hotels   (40552, 8) from hotels.csv


17:25:27 | INFO    | ingest | loaded users    (1340, 5) from users.csv


17:25:31 | INFO    | ingest | wrote flights_clean.csv          (271888, 16)


...

17:25:34 | INFO    | ingest | wrote trips.csv                  (135944, 15)


17:25:35 | INFO    | ingest | wrote users_features.csv         (1340, 19)


17:25:37 | INFO    | ingest | wrote flight_price_model.csv     (271888, 10)


17:25:37 | INFO    | ingest | === ingestion complete: 6 files written ===



files written:
   data\interim\flights_clean.csv
   data\interim\hotels_clean.csv
   data\interim\users_clean.csv
   data\processed\trips.csv
   data\processed\users_features.csv
   data\processed\flight_price_model.csv


## 5. Verify the persisted outputs

In [8]:
def summarize(folder):
    rows = []
    for p in sorted(folder.glob("*.csv")):
        n = sum(1 for _ in open(p, encoding="utf-8")) - 1  # rows minus header
        rows.append((p.name, n, round(p.stat().st_size/1e6, 2)))
    return pd.DataFrame(rows, columns=["file", "rows", "size_MB"])

print("data/interim");    display(summarize(PATHS.interim))
print("data/processed");  display(summarize(PATHS.processed))

data/interim


,file,rows,size_MB
0,flights_clean.csv,271888,42.03
1,hotels_clean.csv,40552,2.80
2,users_clean.csv,1340,0.06


data/processed


,file,rows,size_MB
0,flight_price_model.csv,271888,30.84
1,gender_features.csv,1340,0.22
2,hotel_catalog.csv,9,0.00
3,trips.csv,135944,17.40
4,user_hotel_interactions.csv,9724,0.37
5,users_features.csv,1340,0.20


### 5.1 Load back with typed readers (dates & categories restored)
A bare `pd.read_csv` would return `date` as text. The `load_*` helpers re-apply the dtypes
recorded in `CONFIG`, so downstream models get correctly-typed frames from CSV.

In [9]:
from src.data_ingestion import load_interim, load_processed

fc = load_interim("flights_clean")
price = load_processed("flight_price_model")
print("flights_clean dtypes (note datetime + category):")
print(fc[["date", "from", "flightType", "leg", "price"]].dtypes.to_string())
print("\nflight_price_model:", price.shape)
display(price.head(3))

17:25:39 | INFO    | ingest | loaded flights_clean        (271888, 16) from flights_clean.csv


17:25:40 | INFO    | ingest | loaded flight_price_model   (271888, 10) from flight_price_model.csv


flights_clean dtypes (note datetime + category):
date          datetime64[ns]
from                category
flightType          category
leg                 category
price                float64

flight_price_model: (271888, 10)


,from,to,route,flightType,agency,distance,year,month,day_of_week,price
0,Recife (PE),Florianopolis (SC),Recife (PE) -> Florianopolis (SC),firstClass,FlyingDrops,676.53,2019,9,Thursday,1434.38
1,Florianopolis (SC),Recife (PE),Florianopolis (SC) -> Recife (PE),firstClass,FlyingDrops,676.53,2019,9,Monday,1292.29
2,Brasilia (DF),Florianopolis (SC),Brasilia (DF) -> Florianopolis (SC),firstClass,CloudFy,637.56,2019,10,Thursday,1487.52


## 6. Summary

**Interim (`data/interim/`)** — cleaned, typed, feature-enriched copies of each source table:
`flights_clean`, `hotels_clean`, `users_clean`.

**Processed (`data/processed/`)** — modelling-ready tables:
| File | Grain | Feeds |
|---|---|---|
| `trips.csv` | one round trip | demand / hotel cross-sell |
| `users_features.csv` | one user | gender classification · segmentation · churn |
| `flight_price_model.csv` | one flight leg | flight-price regression |

All tables are stored as **CSV**; load them with `load_interim()` / `load_processed()` so dates and
categoricals are restored. The pipeline is idempotent and re-runnable (`run_pipeline()` or
`python -m src.data_ingestion`), and is the single source of the datasets consumed by the modelling
notebooks next. Data quality is enforced separately by the **validation stage** (`src/validation`),
which gates the raw input before this runs and audits these outputs after.

---
*End of data ingestion.*
